In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import re
import os
import pandas as pd
import multiprocessing
from time import time as timer
from tqdm import tqdm
import numpy as np
from pathlib import Path
from functools import partial
import requests
import urllib
import csv
import json
import shutil


In [ ]:
input_base = '/content/drive/MyDrive/amazon_ml_challenge_VL_75000/student_resource/dataset'  # Adjust if different
train_csv = f'{input_base}/train.csv'
test_csv = f'{input_base}/test.csv'

output_base = '/content/drive/MyDrive/amazon_ml_challenge_VL_75000/image_folder'
os.makedirs(output_base, exist_ok=True)
train_image_folder = f'{output_base}/train_images'
test_image_folder = f'{output_base}/test_images'
os.makedirs(train_image_folder, exist_ok=True)
os.makedirs(test_image_folder, exist_ok=True)

# Utils
import urllib.request
import os
from pathlib import Path
from PIL import Image # Import the Image module from Pillow

# ... (keep the rest of your initial code)

import urllib.request
import os
from pathlib import Path
from PIL import Image, ImageOps # Make sure to import Image and ImageOps

def download_and_resize_image(image_link, savefolder, target_size=(128, 128)):
    """
    Downloads an image, resizes it to a fixed size with padding, and saves it.
    If download fails, it creates a blank placeholder image.
    """
    if not isinstance(image_link, str):
        return None

    filename = Path(image_link).name
    # Ensure the filename ends with .jpg for consistency
    filename = os.path.splitext(filename)[0] + '.jpg'
    image_save_path = os.path.join(savefolder, filename)

    # Skip if the image has already been processed and saved
    if os.path.exists(image_save_path):
        return image_save_path

    temp_path = None
    try:
        # 1. Download the image to a temporary file
        temp_path, _ = urllib.request.urlretrieve(image_link)

        # 2. Open the image with Pillow
        with Image.open(temp_path) as img:
            # Convert to RGB to handle different formats like PNG (with transparency) or GIF
            img = img.convert('RGB')

            # 3. Resize while maintaining aspect ratio (thumbnail) and add padding
            img.thumbnail(target_size, Image.Resampling.LANCZOS)

            # Create a new blank image with the target size and a black background
            padded_img = Image.new("RGB", target_size, "black")

            # Paste the thumbnail onto the center of the blank image
            paste_position = ((target_size[0] - img.width) // 2, (target_size[1] - img.height) // 2)
            padded_img.paste(img, paste_position)

            # 4. Save the final, processed image as JPEG
            padded_img.save(image_save_path, 'JPEG', quality=95)

    except Exception as ex:
        print(f'Warning: Failed to process {image_link}. Creating a blank placeholder. Reason: {ex}')
        try:
            # Create a blank, black image if any step fails
            blank_image = Image.new('RGB', target_size, color='black')
            blank_image.save(image_save_path)
        except Exception as save_ex:
            print(f"FATAL: Could not create blank image for {filename}. Skipping. Reason: {save_ex}")
            return None
    finally:
        # 5. Clean up the temporary file
        if temp_path and os.path.exists(temp_path):
            os.remove(temp_path)

    return image_save_path

# Utils
# def download_image(image_link, savefolder):
#     if isinstance(image_link, str):
#         filename = Path(image_link).name
#         image_save_path = os.path.join(savefolder, filename)
#         if not os.path.exists(image_save_path):
#             try:
#                 urllib.request.urlretrieve(image_link, image_save_path)
#             except Exception as ex:
#                 # --- MODIFICATION STARTS HERE ---
#                 print(f'Warning: Not able to download - {image_link}. Creating a blank placeholder.\n{ex}')
#                 # Create a blank, black 224x224 image
#                 try:
#                     blank_image = Image.new('RGB', (224, 224), color='black')
#                     blank_image.save(image_save_path)
#                 except Exception as save_ex:
#                     print(f"Error: Could not create or save blank image for {filename}. Skipping.\n{save_ex}")
#                     return None # Return None if creating the blank image fails
#                 # --- MODIFICATION ENDS HERE ---
#         return image_save_path
#     return None

# ... (the rest of your script remains the same)
# def download_image(image_link, savefolder):
#     if isinstance(image_link, str):
#         filename = Path(image_link).name
#         image_save_path = os.path.join(savefolder, filename)
#         if not os.path.exists(image_save_path):
#             try:
#                 urllib.request.urlretrieve(image_link, image_save_path)
#             except Exception as ex:
#                 print(f'Warning: Not able to download - {image_link}\n{ex}')
#         return image_save_path
#     return None

# This is your existing function, just change the first line inside it.
def download_images(image_links, download_folder, pool_size=200):
    if not os.path.exists(download_folder):
        os.makedirs(download_folder)

    # --- CHANGE THIS LINE ---
    download_image_partial = partial(download_and_resize_image, savefolder=download_folder, target_size=(128, 128))
    # ----------------------

    with multiprocessing.Pool(pool_size) as pool:
        results = list(tqdm(pool.imap(download_image_partial, image_links), total=len(image_links)))
    return results
# # Process train
df_train = pd.read_csv(train_csv)
# def process_price(value):
#     try:
#         return float(value)
#     except:
#         return "NA"
# df_train['price'] = df_train['price'].apply(process_price)
# df_train = df_train[df_train['price'] != "NA"]
# processed_train_csv = f'{output_base}/processed_train.csv'
# df_train.to_csv(processed_train_csv, index=False)

train_image_links = df_train['image_link'].tolist()
download_images(train_image_links, train_image_folder)

# # Create train JSON
# output_train_json = f'{output_base}/processed_train.json'
# dataset = []
# with open(processed_train_csv, 'r') as csv_file:
#     reader = csv.DictReader(csv_file)
#     for row in reader:
#         image_url = row['image_link']
#         image_filename = os.path.basename(image_url)
#         image_path = os.path.join('train_images', image_filename)  # Relative for dataset
#         if os.path.exists(os.path.join(train_image_folder, image_filename)):
#             catalog_content = row['catalog_content']
#             price = row['price']
#             conversation = {
#                 "messages": [
#                     {"content": f"<image>Based on the product image and this description: {catalog_content}\nWhat is the price?", "role": "user"},
#                     {"content": f"{price}", "role": "assistant"}
#                 ],
#                 "images": [image_path]
#             }
#             dataset.append(conversation)
# with open(output_train_json, 'w') as json_file:
#     json.dump(dataset, json_file, indent=4)

# Process test
df_test = pd.read_csv(test_csv)
# processed_test_csv = f'{output_base}/processed_test.csv'
# df_test.to_csv(processed_test_csv, index=False)

test_image_links = df_test['image_link'].tolist()
download_images(test_image_links, test_image_folder)

# # Optional test JSON
# output_test_json = f'{output_base}/processed_test.json'
# test_dataset = []
# with open(processed_test_csv, 'r') as csv_file:
#     reader = csv.DictReader(csv_file)
#     for row in reader:
#         image_url = row['image_link']
#         image_filename = os.path.basename(image_url)
#         image_path = os.path.join('test_images', image_filename)  # Relative
#         if os.path.exists(os.path.join(test_image_folder, image_filename)):
#             catalog_content = row['catalog_content']
#             conversation = {
#                 "messages": [
#                     {"content": f"<image>Based on the product image and this description: {catalog_content}\nWhat is the price?", "role": "user"}
#                 ],
#                 "images": [image_path],
#                 "sample_id": row['sample_id']
#             }
#             test_dataset.append(conversation)
# with open(output_test_json, 'w') as json_file:
#     json.dump(test_dataset, json_file, indent=4)



print('Dataset created. Search for it in your datasets.')

  4%|▍         | 2892/75000 [38:19<15:55:42,  1.26it/s]Process ForkPoolWorker-60:
Process ForkPoolWorker-161:
Process ForkPoolWorker-419:
Process ForkPoolWorker-418:
Process ForkPoolWorker-391:



KeyboardInterrupt: 

In [ ]:
#!/usr/bin/env python3
"""
Data Preprocessing Script for Amazon ML Challenge 2025 - LLaMA Factory Format
Converts the dataset into a multimodal ShareGPT format using local image paths.
"""

import pandas as pd
import json
import numpy as np
from sklearn.model_selection import train_test_split
import re
import logging
import os
from pathlib import Path
from tqdm import tqdm

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def clean_catalog_content(text):
    """Clean and preprocess catalog content"""
    if pd.isna(text):
        return "No product information available"

    text = str(text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9\s\-\.,\(\)\[\]\&\%\$\/\'\":]', ' ', text)
    text = re.sub(r'(\d+)\s*(pack|pcs|units|count)', r'\1 pack', text, flags=re.IGNORECASE)
    text = text.strip()

    if len(text) == 0:
        return "No product information available"

    return text

def create_price_prediction_entry(catalog_content, image_link, image_folder, price=None, sample_id=None):
    """Create a structured entry for price prediction using the <image> token and a local path."""

    system_content = """You are an expert e-commerce pricing analyst. Given the product image and its details (title, description, quantity), predict the optimal retail price in USD.

Analyze the image for visual cues like quality, branding, and packaging. Analyze the text for specifications and features. Combine both analyses to make your prediction.

Output format: Provide ONLY the predicted price as a float number (e.g., 19.99)."""

    # Clean the catalog content
    cleaned_content = clean_catalog_content(catalog_content)

    # Construct the local image path
    image_path = None
    if pd.notna(image_link) and isinstance(image_link, str):
        try:
            image_filename = Path(image_link).name
            # Create a full, resolved path for the JSON
            image_path = str(Path(image_folder) / image_filename)
        except Exception:
            logger.warning(f"Could not parse image_link to get filename: {image_link}")

    # If the image path is invalid, we cannot proceed with this format
    if not os.path.exists(image_path):
        print("dgsg")
        return None

    # Create user content with the <image> token placeholder
    user_content = f"<image>\nProduct Details: {cleaned_content}"

    entry = {
        "messages": [
            {
                "role": "system",
                "content": system_content
            },
            {
                "role": "user",
                "content": user_content
            }
        ],
        # Add the top-level 'images' key with the local file path
        "images": [image_path]
    }

    if price is not None:
        entry["messages"].append({
            "role": "assistant",
            "content": f"{float(price):.2f}"
        })

    if sample_id is not None:
        entry["sample_id"] = str(sample_id)

    return entry

def preprocess_amazon_dataset(
    train_path='dataset/train.csv',
    test_path='dataset/test.csv',
    output_dir='data',
    image_folder='data/images' # Specify the folder with downloaded images
):
    """Main preprocessing function"""
    logger.info("Starting Amazon ML Challenge dataset preprocessing for multimodal data with local images...")

    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(image_folder, exist_ok=True) # Ensure image folder exists

    try:
        train_df = pd.read_csv(train_path)
        # test_df = pd.read_csv(test_path)
        # logger.info(f"Loaded train: {train_df.shape}, test: {test_df.shape}")
    except FileNotFoundError:
        logger.error(f"Dataset files not found at {train_path} or {test_path}")
        return

    # train_df = train_df.head(40000).copy()
    # test_df = test_df.head(100).copy()

    train_df['price'] = pd.to_numeric(train_df['price'], errors='coerce')
    train_df = train_df.dropna(subset=['catalog_content', 'price', 'image_link'])
    train_df = train_df[train_df['price'] > 0]

    logger.info(f"After cleaning: {len(train_df)} training samples")

    try:
        train_data, val_data = train_test_split(
            train_df, test_size=0.1, random_state=42,
            stratify=pd.qcut(train_df['price'], q=10, labels=False, duplicates='drop')
        )
    except ValueError as e:
        logger.warning(f"Stratified split failed: {e}. Falling back to non-stratified split.")
        train_data, val_data = train_test_split(train_df, test_size=0.1, random_state=42)

    logger.info(f"Split - Train: {len(train_data)}, Validation: {len(val_data)}")

    # --- Convert to ShareGPT format ---
    logger.info("Converting to ShareGPT format with local image paths...")

    # Wrapper function for creating entries
    def process_dataframe(df, desc):
        entries = []
        for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
            entry = create_price_prediction_entry(
                row['catalog_content'],
                row['image_link'],
                image_folder,
                price=row.get('price'), # .get() handles missing price in test_df
                sample_id=row['sample_id']
            )
            if entry:
                entries.append(entry)
        return entries

    train_sharegpt = process_dataframe(train_data, "Processing train data")
    val_sharegpt = process_dataframe(val_data, "Processing val data")
    # test_sharegpt = process_dataframe(test_df, "Processing test data")

    # --- Save datasets ---
    logger.info("Saving processed datasets...")

    def save_json(data, file_path):
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False)

    save_json(train_sharegpt, os.path.join(output_dir, 'price_prediction_train.json'))
    save_json(val_sharegpt, os.path.join(output_dir, 'price_prediction_val.json'))
    # save_json(test_sharegpt, os.path.join(output_dir, 'price_prediction_test.json'))

    logger.info("Dataset preprocessing completed!")
    logger.info(f"Files created in {output_dir}:")
    logger.info(f"  - price_prediction_train.json: {len(train_sharegpt)} samples")
    logger.info(f"  - price_prediction_val.json: {len(val_sharegpt)} samples")
    # logger.info(f"  - price_prediction_test.json: {len(test_sharegpt)} samples")

    logger.info("\nSample training data entry:")
    if train_sharegpt:
        print(json.dumps(train_sharegpt[0], indent=2))

if __name__ == "__main__":
    # NOTE: This script assumes you have already downloaded the images
    # into a folder, specified by the `image_folder` argument.
    preprocess_amazon_dataset(
        train_path='/content/drive/MyDrive/amazon_ml_challenge_VL_75000/student_resource/dataset/train.csv',
        test_path='/content/drive/MyDrive/amazon_ml_challenge_VL_75000/student_resource/dataset/test.csv',
        output_dir='/content/drive/MyDrive/amazon_ml_challenge_VL_75000/Output',
        image_folder='/content/drive/MyDrive/amazon_ml_challenge_VL_75000/image_folder/train_images' # Assumes images are in 'dataset/images/'
    )

In [ ]:
#!/usr/bin/env python3
"""
Data Preprocessing Script for Amazon ML Challenge 2025 - LLaMA Factory Format
Converts the dataset into a multimodal ShareGPT format using local image paths.
"""

import pandas as pd
import json
import numpy as np
from sklearn.model_selection import train_test_split
import re
import logging
import os
from pathlib import Path
from tqdm import tqdm

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def clean_catalog_content(text):
    """Clean and preprocess catalog content"""
    if pd.isna(text):
        return "No product information available"

    text = str(text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9\s\-\.,\(\)\[\]\&\%\$\/\'\":]', ' ', text)
    text = re.sub(r'(\d+)\s*(pack|pcs|units|count)', r'\1 pack', text, flags=re.IGNORECASE)
    text = text.strip()

    if len(text) == 0:
        return "No product information available"

    return text

def create_price_prediction_entry(catalog_content, image_link, image_folder, price=None, sample_id=None):
    """Create a structured entry for price prediction using the <image> token and a local path."""

    system_content = """You are an expert e-commerce pricing analyst. Given the product image and its details (title, description, quantity), predict the optimal retail price in USD.

Analyze the image for visual cues like quality, branding, and packaging. Analyze the text for specifications and features. Combine both analyses to make your prediction.

Output format: Provide ONLY the predicted price as a float number (e.g., 19.99)."""

    # Clean the catalog content
    cleaned_content = clean_catalog_content(catalog_content)

    # Construct the local image path
    image_path = None
    if pd.notna(image_link) and isinstance(image_link, str):
        try:
            image_filename = Path(image_link).name
            # Create a full, resolved path for the JSON
            image_path = str(Path(image_folder) / image_filename)
        except Exception:
            logger.warning(f"Could not parse image_link to get filename: {image_link}")

    # If the image path is invalid, we cannot proceed with this format
    if not os.path.exists(image_path):
        print("dgsg")
        return None

    # Create user content with the <image> token placeholder
    user_content = f"<image>\nProduct Details: {cleaned_content}"

    entry = {
        "messages": [
            {
                "role": "system",
                "content": system_content
            },
            {
                "role": "user",
                "content": user_content
            }
        ],
        # Add the top-level 'images' key with the local file path
        "images": [image_path]
    }

    if price is not None:
        entry["messages"].append({
            "role": "assistant",
            "content": f"{float(price):.2f}"
        })

    if sample_id is not None:
        entry["sample_id"] = str(sample_id)

    return entry

def preprocess_amazon_dataset(
    train_path='dataset/train.csv',
    test_path='dataset/test.csv',
    output_dir='data',
    image_folder='data/images' # Specify the folder with downloaded images
):
    """Main preprocessing function"""
    logger.info("Starting Amazon ML Challenge dataset preprocessing for multimodal data with local images...")

    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(image_folder, exist_ok=True) # Ensure image folder exists

    try:
        # train_df = pd.read_csv(train_path)
        test_df = pd.read_csv(test_path)
        # logger.info(f"Loaded train: {train_df.shape}, test: {test_df.shape}")
    except FileNotFoundError:
        logger.error(f"Dataset files not found at {train_path} or {test_path}")
        return

    # train_df = train_df.head(100).copy()
    # test_df = test_df.head(100).copy()

    # train_df['price'] = pd.to_numeric(train_df['price'], errors='coerce')
    # train_df = train_df.dropna(subset=['catalog_content', 'price', 'image_link'])
    # train_df = train_df[train_df['price'] > 0]

    # logger.info(f"After cleaning: {len(train_df)} training samples")

    # try:
    #     train_data, val_data = train_test_split(
    #         train_df, test_size=0.1, random_state=42,
    #         stratify=pd.qcut(train_df['price'], q=10, labels=False, duplicates='drop')
    #     )
    # except ValueError as e:
    #     logger.warning(f"Stratified split failed: {e}. Falling back to non-stratified split.")
    #     train_data, val_data = train_test_split(train_df, test_size=0.1, random_state=42)

    # logger.info(f"Split - Train: {len(train_data)}, Validation: {len(val_data)}")

    # --- Convert to ShareGPT format ---
    logger.info("Converting to ShareGPT format with local image paths...")

    # Wrapper function for creating entries
    def process_dataframe(df, desc):
        entries = []
        for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
            entry = create_price_prediction_entry(
                row['catalog_content'],
                row['image_link'],
                image_folder,
                price=row.get('price'), # .get() handles missing price in test_df
                sample_id=row['sample_id']
            )
            if entry:
                entries.append(entry)
        return entries

    # train_sharegpt = process_dataframe(train_data, "Processing train data")
    # val_sharegpt = process_dataframe(val_data, "Processing val data")
    test_sharegpt = process_dataframe(test_df, "Processing test data")

    # --- Save datasets ---
    logger.info("Saving processed datasets...")

    def save_json(data, file_path):
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False)

    # save_json(train_sharegpt, os.path.join(output_dir, 'price_prediction_train.json'))
    # save_json(val_sharegpt, os.path.join(output_dir, 'price_prediction_val.json'))
    save_json(test_sharegpt, os.path.join(output_dir, 'price_prediction_test.json'))

    logger.info("Dataset preprocessing completed!")
    logger.info(f"Files created in {output_dir}:")
    # logger.info(f"  - price_prediction_train.json: {len(train_sharegpt)} samples")
    # logger.info(f"  - price_prediction_val.json: {len(val_sharegpt)} samples")
    logger.info(f"  - price_prediction_test.json: {len(test_sharegpt)} samples")

    logger.info("\nSample training data entry:")
    if test_sharegpt:
        print(json.dumps(test_sharegpt[0], indent=2))

if __name__ == "__main__":
    # NOTE: This script assumes you have already downloaded the images
    # into a folder, specified by the `image_folder` argument.
    preprocess_amazon_dataset(
        train_path='/content/drive/MyDrive/amazon_ml_challenge_VL_75000/student_resource/dataset/train.csv',
        test_path='/content/drive/MyDrive/amazon_ml_challenge_VL_75000/student_resource/dataset/test.csv',
        output_dir='/content/drive/MyDrive/amazon_ml_challenge_VL_75000/Output',
        image_folder='/content/drive/MyDrive/amazon_ml_challenge_VL_75000/image_folder/test_images'
    )

In [ ]:
!git clone https://github.com/hiyouga/LLaMA-Factory.git


In [ ]:
dataset={
  "pricing_train": {
    "file_name": "price_prediction_train.json",
    "formatting": "sharegpt",
    "columns": {
      "messages": "messages",
      "images": "images"
    },
    "tags": {
      "role_tag": "role",
      "content_tag": "content",
      "user_tag": "user",
      "assistant_tag": "assistant",
      "system_tag": "system"
    }
  },
  "pricing_val": {
    "file_name": "price_prediction_val.json",
    "formatting": "sharegpt",
    "columns": {
      "messages": "messages",
      "images": "images"
    },
    "tags": {
      "role_tag": "role",
      "content_tag": "content",
      "user_tag": "user",
      "assistant_tag": "assistant",
      "system_tag": "system"
    }
  }
}

metadata_file = '/kaggle/working/dataset_info.json'
with open(metadata_file, 'w', encoding='utf-8') as f:
    json.dump(dataset, f, indent=2)

In [ ]:
%%bash
cp /content/drive/MyDrive/amazon_ml_challenge_VL_75000/Output/dataset_info.json /content/drive/MyDrive/amazon_ml_challenge_VL_75000/LLaMA-Factory/data
cp /content/drive/MyDrive/amazon_ml_challenge_VL_75000/Output/price_prediction_train.json /content/drive/MyDrive/amazon_ml_challenge_VL_75000/LLaMA-Factory/data
cp /content/drive/MyDrive/amazon_ml_challenge_VL_75000/Output/price_prediction_val.json /content/drive/MyDrive/amazon_ml_challenge_VL_75000/LLaMA-Factory/data

In [ ]:
%cd /content/drive/MyDrive/amazon_ml_challenge_VL_75000/LLaMA-Factory

In [ ]:
!pip install -r requirements.txt -q

In [ ]:
!pip install "deepspeed>=0.10.0,<=0.16.9" -q

In [ ]:
!pip install -e ".[torch, metrics]" -q

In [ ]:
!pip install bitsandbytes -q

In [ ]:
!pip install accelerate tensorboard peft -q

In [ ]:
import json
import logging

logger = logging.getLogger(__name__)
DS_CONFIG_FILE = '/content/drive/MyDrive/amazon_ml_challenge_VL_75000/ds_zero2.json' # Renamed for clarity

def write_deepspeed_config():
    """Write a faster DeepSpeed Zero-2 config file for T4 GPUs."""
    ds_config = {
        "zero_optimization": {
            "stage": 2,  # Use Stage 2 for less communication overhead
            "offload_optimizer": {"device": "none"},
            "overlap_comm": True,
            "contiguous_gradients": True,
            "reduce_bucket_size": "auto",
            "stage3_prefetch_bucket_size": 0, # Not used in Stage 2
            "stage3_param_persistence_threshold": 0, # Not used in Stage 2
        },
        "fp16": { # Use fp16 instead of bf16 for T4 GPUs
            "enabled": True,
            "loss_scale": 0,
            "loss_scale_window": 1000,
            "hysteresis": 2,
            "min_loss_scale": 1
        },
        "gradient_accumulation_steps": "auto",
        "gradient_clipping": "auto",
        "steps_per_print": 2000,
        "train_batch_size": "auto",
        "train_micro_batch_size_per_gpu": "auto",
        "wall_clock_breakdown": False
    }
    with open(DS_CONFIG_FILE, 'w') as f:
        json.dump(ds_config, f, indent=2)
    logger.info(f"DeepSpeed config written to {DS_CONFIG_FILE}")

# Don't forget to update the file path in your main script
# and in the Llama Factory config.
write_deepspeed_config()

In [ ]:
!cp /content/drive/MyDrive/amazon_ml_challenge_VL_75000/ds_zero2.json /content/drive/MyDrive/amazon_ml_challenge_VL_75000/LLaMA-Factory/data
!cp /content/drive/MyDrive/amazon_ml_challenge_VL_75000/ds_zero2.json /content/drive/MyDrive/amazon_ml_challenge_VL_75000/LLaMA-Factory

In [ ]:
import json
import logging

logger = logging.getLogger(__name__)

JSON_CONFIG_FILE = "config.json"

def write_json_config():
    """Write Llama Factory JSON config without flash attention"""
    config = {
        "model_name_or_path": "Qwen/Qwen2.5-VL-3B-Instruct",
        "stage": "sft",
        "do_train": True,
        "finetuning_type": "lora",
        "lora_rank": 32,
        "lora_alpha": 64,
        "lora_dropout": 0.1,
        "lora_target": "all",

        "dataset": "pricing_train",
        "dataset_dir": "./data",
        "eval_dataset": "pricing_val",
        "template": "qwen2_vl",
        "cutoff_len": 1024,
        "train_on_prompt": False,

        "quantization_bit": 4,
        "quantization_type": "nf4",

        "preprocessing_num_workers": 8,
        "per_device_train_batch_size": 4,
        "per_device_eval_batch_size": 2,
        "gradient_accumulation_steps": 16,
        "num_train_epochs": 2,
        "learning_rate": 1e-5,
        "max_grad_norm": 1.0,
        "logging_steps": 10,
        "save_steps": 30,
        "save_strategy": "steps",
        "save_total_limit": 20,
        "eval_steps": 30,
        "eval_strategy": "steps",
        "load_best_model_at_end": False,
        "output_dir": "qwen_out",
        "overwrite_output_dir": True,
        "plot_loss": True,

        "optim": "adamw_bnb_8bit",
        "lr_scheduler_type": "cosine",
        "warmup_steps": 10,
        "weight_decay": 0.1,

        "fp16": True,  # Add this from step 1
        "flash_attn": "sdpa", # Add this line
        "tf32": False,
        "gradient_checkpointing": False,
        "packing": True,

        "ddp_backend": "nccl",
        "seed": 42,
        "report_to": "tensorboard",
        "deepspeed": "./ds_zero2.json"
    }

    with open(JSON_CONFIG_FILE, "w") as f:
        json.dump(config, f, indent=4)
    logger.info(f"JSON config written to {JSON_CONFIG_FILE}")


write_json_config()


In [ ]:
!llamafactory-cli train config.json

In [ ]:

%%bash
pip install qwen-vl-utils



In [ ]:
# ==========================================
# SIMPLE MULTI-GPU INFERENCE FOR KAGGLE 2x T4
# ==========================================
# This approach uses DataParallel which is the simplest way to use both GPUs

import os
import json
import torch
import pandas as pd
import numpy as np
import re
import logging
from pathlib import Path
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from peft import PeftModel
from tqdm import tqdm
import shutil
import gc

# Configuration
MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"
OUTPUT_DIR = "/content/drive/MyDrive/amazon_ml_challenge_VL_75000/LLaMA-Factory/qwen_out"
DATA_DIR = "./data"
MERGED_MODEL_DIR = "/content/drive/MyDrive/amazon_ml_challenge_VL_75000/Output/merged_model"

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Flash attention fix
def fixed_get_imports(filename):
    if "modeling_qwen2" in filename:
        return ["torch", "torch.nn", "transformers", "torch.nn.functional"]
    else:
        return get_imports_orig(filename)

try:
    import transformers.dynamic_module_utils
    get_imports_orig = transformers.dynamic_module_utils.get_imports
    transformers.dynamic_module_utils.get_imports = fixed_get_imports
except:
    pass

def calculate_smape(predictions, targets):
    predictions = np.array(predictions)
    targets = np.array(targets)
    denominator = (np.abs(targets) + np.abs(predictions)) / 2
    mask = denominator != 0
    if mask.sum() == 0:
        return 100.0
    return np.mean(np.abs(predictions[mask] - targets[mask]) / denominator[mask]) * 100

def extract_price_from_text(text):
    try:
        match = re.search(r'\b\d+\.\d{2}\b', text.strip())
        if match:
            return float(match.group(0))
        match = re.search(r'\b\d+\.\d+\b', text.strip())
        if match:
            return float(match.group(0))
        match = re.search(r'\b\d+\b', text.strip())
        if match:
            return float(match.group(0))
    except:
        pass
    return 0.0

def evaluate_checkpoint(checkpoint_path):
    """Updated function to work with Qwen2.5-VL model"""
    logger.info(f"Evaluating checkpoint: {checkpoint_path}")

    # Use AutoProcessor instead of AutoTokenizer
    processor = AutoProcessor.from_pretrained(MODEL_NAME)

    # Use Qwen2_5_VLForConditionalGeneration instead of AutoModelForCausalLM
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="cuda:0",  # Use only first GPU for evaluation
        trust_remote_code=True,
        attn_implementation="sdpa"
    )

    model = PeftModel.from_pretrained(model, checkpoint_path)
    model.eval()

    with open(os.path.join(DATA_DIR, 'price_prediction_val.json'), 'r') as f:
        val_data = json.load(f)

    predictions = []
    ground_truths = []

    for item in tqdm(val_data[:50], desc="Validating"):
        # Create messages in the VL format
        system_msg = item["messages"][0]
        user_msg = item["messages"][1]
        image_path = item["images"][0]

        new_user_content = [
            {"type": "image", "image": image_path},
            {"type": "text", "text": user_msg["content"].replace("<image>", "").strip()}
        ]

        new_messages = [
            {
                "role": "system",
                "content": [{"type": "text", "text": item['messages'][0]['content']}]
            },
            {"role": "user", "content": new_user_content}
        ]
        # messages = [
        #     {
        #         "role": "system",
        #         "content": [{"type": "text", "text": item['messages'][0]['content']}]
        #     },
        #     {
        #         "role": "user",
        #         "content": [{"type": "text", "text": item['messages'][1]['content']}]
        #     }
        # ]

        gt_text = item['messages'][2]['content']
        gt_price = extract_price_from_text(gt_text)
        ground_truths.append(gt_price)

        # Use processor instead of tokenizer
        text = processor.apply_chat_template(new_messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=20, do_sample=False, temperature=0.0)

        generated_text = processor.batch_decode(
            outputs[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True
        )[0]
        predicted_price = extract_price_from_text(generated_text)
        predictions.append(predicted_price)

    smape = calculate_smape(predictions, ground_truths)

    del model, processor
    torch.cuda.empty_cache()
    gc.collect()

    return smape


def select_best_checkpoint():
    checkpoint_dir = Path(OUTPUT_DIR)
    checkpoints = [p for p in checkpoint_dir.iterdir() if p.is_dir() and p.name.startswith('checkpoint-')]
    print(checkpoints)

    if not checkpoints:
        logger.warning("No checkpoints found! Using final model.")
        return str(checkpoint_dir)

    print(f"Found {len(checkpoints)} checkpoints")
    best_smape = float('inf')
    best_checkpoint = None

    for checkpoint in checkpoints:
        try:
            smape = evaluate_checkpoint(str(checkpoint))
            logger.info(f"{checkpoint.name}: SMAPE = {smape:.4f}%")


            if smape < best_smape:
                best_smape = smape
                best_checkpoint = checkpoint
        except Exception as e:

            logger.error(f"Error evaluating {checkpoint}: {e}")


    return str(best_checkpoint) if best_checkpoint else None

def merge_and_export_model(best_checkpoint_path):
    logger.info(f"Merging model from {best_checkpoint_path}")

    os.makedirs(MERGED_MODEL_DIR, exist_ok=True)

    processor = AutoProcessor.from_pretrained(MODEL_NAME)
    base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="cpu",
        trust_remote_code=True
    )

    model = PeftModel.from_pretrained(base_model, best_checkpoint_path)
    merged_model = model.merge_and_unload()

    merged_model.save_pretrained(MERGED_MODEL_DIR, safe_serialization=True, max_shard_size="2GB")
    processor.save_pretrained(MERGED_MODEL_DIR)

    logger.info(f"Merged model saved to {MERGED_MODEL_DIR}")

    del base_model, model, merged_model
    torch.cuda.empty_cache()
    gc.collect()


# ==========================================
# MAIN EXECUTION
# ==========================================

print("🚀 Starting checkpoint evaluation and inference pipeline...")

# Step 1: Select best checkpoint
print("\n📊 Step 1: Selecting Best Checkpoint")
best_checkpoint = select_best_checkpoint()
print(f"✅ Selected checkpoint: {best_checkpoint}")

if best_checkpoint:
    print(f"✅ Selected checkpoint: {best_checkpoint}")

    # Step 2: Merge and export
    print("\n🔄 Step 2: Merging and Exporting Model")
    merge_and_export_model(best_checkpoint)



else:
    print("❌ No valid checkpoints found!")


In [ ]:
!deepspeed --num_gpus=2 ./VL_inference.py